In [362]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

## Data Preprocessing
First We read the training data and perform preprocessing.

In [363]:
df = pd.read_csv("train.csv")
print(f"Number of rows: {df.shape[0]}     Number of columns: {df.shape[1]}")

Number of rows: 891     Number of columns: 12


In [364]:
df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
601,602,0,3,"Slabenoff, Mr. Petco",male,NaN,0,0,349214,7.8958,NaN,S
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C
465,466,0,3,"Goncalves, Mr. Manuel Estanslas",male,38.0,0,0,SOTON/O.Q. 3101306,7.0500,NaN,S
82,83,1,3,"McDermott, Miss. Brigdet Delia",female,NaN,0,0,330932,7.7875,NaN,Q
332,333,0,1,"Graham, Mr. George Edward",male,38.0,0,1,PC 17582,153.4625,C91,S
438,439,0,1,"Fortune, Mr. Mark",male,64.0,1,4,19950,263.0000,C23 C25 C27,S
749,750,0,3,"Connaghton, Mr. Michael",male,31.0,0,0,335097,7.7500,NaN,Q
698,699,0,1,"Thayer, Mr. John Borland",male,49.0,1,1,17421,110.8833,C68,C
616,617,0,3,"Danbom, Mr. Ernst Gilbert",male,34.0,1,1,347080,14.4000,NaN,S
448,449,1,3,"Baclini, Miss. Marie Catherine",female,5.0,2,1,2666,19.2583,NaN,C


In [365]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


The full names in the `Name` column in the dataset is not very effective in our prediction model but we extract the title from it. Then we replace the uncommon titles with the title `Rare`

In [366]:
df["Title"] = df["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()
df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [367]:
df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

rare_titles = ["Dr", "Rev", "Col", "Major", "Don", "Lady", "Sir", "Capt", "the Countess", "Jonkheer"]
df["Title"] = df["Title"].replace(rare_titles, "Rare")

df = df.drop(columns = ["Name"])

df["Title"].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

Now we convert categorical data to numerical data using One Hot Encoding.

In [368]:
from sklearn.preprocessing import OneHotEncoder

# Before converting Embarked we should fill out the 2 missing values. We use mode of the column here.
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore")
new_columns = encoder.fit_transform(df[["Sex", "Embarked", "Title"]])
# Now we add the new columns to the dataset
df[encoder.get_feature_names_out()] = new_columns
# Drop the categorical columns
df.drop(columns = ["Sex", "Embarked", "Title"], inplace = True)

The Ticket Column does not provide useful information so we drop it. We also drop PassengerId.

In [369]:
df = df.drop(columns = ["Ticket", "PassengerId"])


Let's take a look at our dataset again.

In [370]:
df.sample(10)

,Survived,Pclass,Age,SibSp,Parch,Fare,Cabin,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
705,0,2,39.0,0,0,26.0000,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
405,0,2,34.0,1,0,21.0000,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
862,1,1,48.0,0,0,25.9292,D17,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
421,0,3,21.0,0,0,7.7333,NaN,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
232,0,2,59.0,0,0,13.5000,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
742,1,1,21.0,2,2,262.3750,B57 B59 B63 B66,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
816,0,3,23.0,0,0,7.9250,NaN,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
563,0,3,NaN,0,0,8.0500,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
848,0,2,28.0,0,1,33.0000,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
439,0,2,31.0,0,0,10.5000,NaN,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


Now we check for Null values

In [371]:
df.isna().sum()

Survived          0
Pclass            0
Age             177
SibSp             0
Parch             0
Fare              0
Cabin           687
Sex_female        0
Sex_male          0
Embarked_C        0
Embarked_Q        0
Embarked_S        0
Title_Master      0
Title_Miss        0
Title_Mr          0
Title_Mrs         0
Title_Rare        0
dtype: int64

The column `Cabin` has so many null values so we drop the whole column. and for the null values of column `Age` we train a model to predict the age.

In [372]:
# First we drop the Cabin column
df = df.drop(columns = ["Cabin"])

# Now we need to deal with the Null values in the "Age" column. first we add a new column to indicate
# the rows with missing "Age" column
df["Age_missing"] = df["Age"].isna().astype(int)

Now we train a Regression Decision Tree to predict the missing "Age" values.
First we split the rows into training and test data. the rows with missing "Age" values will be the test data that we want to predict and the rest are the training data.

In [373]:
# Separate the rows based on missing "Age"
predict_age_train = df[df["Age"].notna()]
predict_age_test = df[df["Age"].isna()]
# Separate the labels and the features. Here the label is the "Age" column not the "Survived"
# We also drop "Survived" column so the model also works for test data
X_train = predict_age_train.drop(columns = ["Age", "Survived"])
y_train = predict_age_train["Age"]

X_test = predict_age_test.drop(columns = ["Age", "Survived"])


In [374]:
# Now we train a Regression Decision Tree
reg_decision_tree = DecisionTreeRegressor(max_depth = 5, min_samples_leaf = 5, random_state = 10)
reg_decision_tree.fit(X_train, y_train)
age_predictions = reg_decision_tree.predict(X_test)

df.loc[df["Age"].isna(), "Age"] = age_predictions.round(1)

Let's check the missing values in each column again.

In [375]:
df.isna().sum()

Survived        0
Pclass          0
Age             0
SibSp           0
Parch           0
Fare            0
Sex_female      0
Sex_male        0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
Title_Master    0
Title_Miss      0
Title_Mr        0
Title_Mrs       0
Title_Rare      0
Age_missing     0
dtype: int64

Now the dataset is ready to train a Decision Tree for classification. this time the labels are the `Survived` column.

In [376]:
X = df.drop(columns = ["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 10)

# Decision Tree Model
model = DecisionTreeClassifier(criterion = "gini", max_depth = 5, min_samples_leaf = 5, random_state = 10)
model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,10
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


Now we perform the same steps on the test data

In [380]:
df_test = pd.read_csv("test.csv")
df_test_original = df_test

# First we extract and replace the titles from column "Name"
df_test["Title"] = df_test["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()
df_test["Title"] = df_test["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})
# There is a new title in test data "Dona" we add it to the rare_titles
rare_titles.append("Dona")
df_test["Title"] = df_test["Title"].replace(rare_titles, "Rare")
df_test = df_test.drop(columns = "Name")

# Convert categorical data to numerical
test_new_columns = encoder.transform(df_test[["Sex", "Embarked", "Title"]])
df_test[encoder.get_feature_names_out()] = test_new_columns

# We also drop the Ticket and Cabin column
df_test = df_test.drop(columns = ["PassengerId", "Sex", "Ticket", "Cabin", "Embarked", "Title"])

# Now we fill out the "Age" null values the same way we did before
# But before we see that there is one row missing the value of "Fare" we use KNNImputer to fill it out
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors = 10)
temp_imputed = imputer.fit_transform(df_test)
df_test["Fare"] = temp_imputed[:, df_test.columns.get_loc("Fare")]
# Now we fill out the missing values in Age column but before we add a new column for missing age
df_test["Age_missing"] = df_test["Age"].isna().astype(int)

X_test_age = df_test[df_test["Age"].isna()]
X_test_age = X_test_age.drop(columns = ["Age"])
df_test.loc[df_test["Age"].isna(), "Age"] = reg_decision_tree.predict(X_test_age).round(1)

Let's take a look at the test data.

In [381]:
df_test.sample(10)

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Age_missing
248,2,29.0,1,0,26.00000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0
152,3,60.5,0,0,12.73999,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
263,3,1.0,1,1,12.18330,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0
37,3,21.0,0,0,8.66250,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0
405,2,20.0,0,0,13.86250,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
193,2,61.0,0,0,12.35000,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0
339,3,5.7,0,0,7.22920,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
70,3,24.0,0,0,7.75000,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
71,3,21.0,0,0,7.89580,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
341,3,32.0,0,0,7.57920,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0


In [386]:
test_predications = model.predict(df_test).astype(int)

submission_file = pd.DataFrame({
    "PassengerId": df_test_original["PassengerId"],
    "Survived": test_predications
})

submission_file.to_csv("submission.csv", index = False)